# Loading with Podio

In [15]:
!cd /global/homes/d/danieltm & pwd
!export ALRB_CONT_SWTYPE=shifter

/global/cfs/cdirs/m4958/usr/danieltm/ColliderML/software/colliderml_dev/notebooks/physics


In [7]:
!source /global/cfs/projectdirs/atlas/scripts/setupATLAS.sh

In [16]:
!setupATLAS -c el9

/bin/bash: setupATLAS: command not found


In [9]:
!source /cvmfs/sft.cern.ch/lcg/views/setupViews.sh LCG_107 x86_64-el8-gcc11-opt

In [10]:
import torch

In [18]:
%%bash
cd $HOME
pwd
export ALRB_CONT_SWTYPE=shifter
source /global/cfs/projectdirs/atlas/scripts/setupATLAS.sh
setupATLAS -c el9
source /cvmfs/sft.cern.ch/lcg/views/setupViews.sh LCG_107 x86_64-el9-gcc13-opt

/global/homes/d/danieltm
Info: /cvmfs mounted; do 'setupATLAS -d -c ...' to skip default mounts.
------------------------------------------------------------------------------
Shifter: 18.03.6_20211205_d5d697c-8.nersc
Host: Linux, SUSE Linux Enterprise Server 15 SP5, x86_64, 5.14.21-150500.55.97_13.0.78-cray_shasta_c
From: /usr/bin/shifter
ContainerType: atlas-default
shifter    -V /tmp/danieltm:/tmp/danieltm --clearenv --image=registry.cern.ch/atlasadc/atlas-grid-almalinux9 -V /tmp/danieltm/.alrb/container/shifter/home.KtkzBc:/alrb  -m cvmfs  -V /global/u2/d/danieltm:/srv  /bin/bash --rcfile /alrb/.bashrc
------------------------------------------------------------------------------


In [1]:
!source /cvmfs/sw-nightlies.hsf.org/key4hep/setup.sh

Unsupported OS or OS couldn't be correctly detected, aborting...
Supported OSes are: AlmaLinux/RockyLinux/RHEL 9 and Ubuntu 24.04


## Try with key4hep

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uproot
import sys
import seaborn as sns
from tqdm import tqdm
import networkx as nx

import atlasify as atl
from particle import Particle
atl.ATLAS = "ColliderML"

sys.path.append("/global/cfs/cdirs/m4958/usr/danieltm/ColliderML/software/OtherLibraries/pyedm4hep")
from pyedm4hep import EDM4hepEvent, EDM4hepEventBatch

## Loading

In [11]:
# Define conservative detector parameters
detector_params = {
    'tracking_radius': 1080,    # in mm
    'tracking_z_max': 3030,     # in mm
    'energy_threshold': 0.     # in GeV
}

edm_input_file = "/global/cfs/cdirs/m4958/data/ColliderML/simulation/single_particle_pilot/single_electron_loguniform/v1/runs/0/edm4hep.root"
batch = EDM4hepEventBatch(edm_input_file, events=(0, 1000), full_load=False, detector_params=detector_params)

In [12]:
tracker_hits = batch.get_tracker_hits_df()
calo_hits = batch.get_calo_hits_df()
calo_contributions = batch.get_calo_contributions_df()
particles = batch.get_particles_df()


Augmenting particle hit counts with tracker hits


In [8]:
calo_digi_file = "/global/cfs/cdirs/m4958/data/ColliderML/simulation/single_particle_pilot/single_electron_loguniform/v1/runs/0/edm4hep_digitized.root"

In [15]:
calo_digi_root = uproot.open(calo_digi_file)
calo_digi_event = calo_digi_root['events']
calo_digi_event.keys()


['digiECalBarrelCollection',
 'digiECalBarrelCollection/digiECalBarrelCollection.cellID',
 'digiECalBarrelCollection/digiECalBarrelCollection.energy',
 'digiECalBarrelCollection/digiECalBarrelCollection.energyError',
 'digiECalBarrelCollection/digiECalBarrelCollection.time',
 'digiECalBarrelCollection/digiECalBarrelCollection.position.x',
 'digiECalBarrelCollection/digiECalBarrelCollection.position.y',
 'digiECalBarrelCollection/digiECalBarrelCollection.position.z',
 'digiECalBarrelCollection/digiECalBarrelCollection.type',
 'digiECalEndcapCollection',
 'digiECalEndcapCollection/digiECalEndcapCollection.cellID',
 'digiECalEndcapCollection/digiECalEndcapCollection.energy',
 'digiECalEndcapCollection/digiECalEndcapCollection.energyError',
 'digiECalEndcapCollection/digiECalEndcapCollection.time',
 'digiECalEndcapCollection/digiECalEndcapCollection.position.x',
 'digiECalEndcapCollection/digiECalEndcapCollection.position.y',
 'digiECalEndcapCollection/digiECalEndcapCollection.position.z',

## Roadmap

1. What is the structure of calo digi file?
2. Can we match truth calo cells to digitised cells?
3. 

In [ ]:
# Test loading digitized calorimeter hits
digi_hits = batch.get_digi_calo_hits_df()

print(f"Loaded {len(digi_hits)} digitized calo hits")
print(f"\nColumns: {list(digi_hits.columns)}")
print(f"\nDetectors: {digi_hits['detector'].unique()}")
print(f"\nFirst few rows:")
print(digi_hits.head())

# Verify all expected columns are present
expected_cols = ['cellID', 'energy', 'energy_error', 'time', 'x', 'y', 'z', 'type', 
                 'detector', 'r', 'R', 'phi', 'theta', 'eta', 'event_id']
missing_cols = set(expected_cols) - set(digi_hits.columns)
if missing_cols:
    print(f"\nWARNING: Missing columns: {missing_cols}")
else:
    print(f"\n✓ All expected columns present")
